# Модуль 4 · Урок 2 — DRF: ViewSets, Routers, автентифікація та permissions

👋 В Уроці 1 ми зробили CRUD через `APIView` та generic views. Тепер — **продакшн-інструменти
DRF**: ViewSets і Routers (менше коду), фільтрація/пошук/сортування, пагінація, **автентифікація**
(хто ти?) і **permissions** (що тобі можна?), а також коректна обробка помилок.

**Передумова:** [Урок 1](lektsiya-1-django-rest-framework-osnovy.ipynb).

### Що ви вмітимете після уроку
- зводити CRUD до кількох рядків через **ModelViewSet + Router**;
- додавати **фільтрацію, пошук, сортування** та **пагінацію**;
- пояснити типи **автентифікації** (Session, Basic, Token) і налаштувати **Token Auth**;
- керувати доступом через **permissions** (вбудовані + кастомні);
- повертати правильні **status codes** і зрозумілі помилки.

## 1. ViewSets та ModelViewSet

**ViewSet** об'єднує логіку над одним ресурсом в **один клас** замість кількох view. Замість
методів `get`/`post` тут — «дії»: `list`, `retrieve`, `create`, `update`, `partial_update`,
`destroy`. А **`ModelViewSet`** дає **весь CRUD одразу**:

```python
# articles/views.py
from rest_framework import viewsets
from .models import Article
from .serializers import ArticleSerializer

class ArticleViewSet(viewsets.ModelViewSet):
    queryset = Article.objects.all()
    serializer_class = ArticleSerializer
```

Цей один клас = **6 ендпоінтів CRUD**:

| Дія (метод ViewSet) | HTTP + URL |
|---------------------|-----------|
| `list` | `GET /articles/` |
| `create` | `POST /articles/` |
| `retrieve` | `GET /articles/{id}/` |
| `update` | `PUT /articles/{id}/` |
| `partial_update` | `PATCH /articles/{id}/` |
| `destroy` | `DELETE /articles/{id}/` |

Можна додати **власну дію** декоратором `@action` — напр. «опублікувати статтю»:
```python
from rest_framework.decorators import action
from rest_framework.response import Response

class ArticleViewSet(viewsets.ModelViewSet):
    queryset = Article.objects.all()
    serializer_class = ArticleSerializer

    @action(detail=True, methods=["post"])        # POST /articles/{id}/publish/
    def publish(self, request, pk=None):
        article = self.get_object()
        article.is_published = True
        article.save()
        return Response({"status": "опубліковано"})
```

## 2. Routers — автоматичні маршрути

Для ViewSet не треба виписувати кожен `path()` — **Router** сам згенерує всі URL CRUD:

```python
# articles/urls.py
from rest_framework.routers import DefaultRouter
from .views import ArticleViewSet

router = DefaultRouter()
router.register(r"articles", ArticleViewSet)   # реєструємо ресурс

urlpatterns = router.urls                       # усі 6 маршрутів створено автоматично
```

Router сам зробить `/articles/`, `/articles/{id}/`, а також `/articles/{id}/publish/` для нашої
`@action`.

| Router | Особливість |
|--------|-------------|
| `DefaultRouter` | + коренева сторінка API зі списком ендпоінтів (зручно для розробки) |
| `SimpleRouter` | те саме, але **без** кореневої сторінки (частіше для продакшену) |

> 🎯 **Питання: APIView vs ViewSet?** `APIView`/generic — один клас = один URL (ви керуєте
> маршрутами). `ViewSet` + `Router` — один клас = **увесь CRUD** з авто-маршрутами. ViewSet
> зручний для стандартних ресурсів; коли логіка нестандартна — беруть generic/APIView.

## 3. Фільтрація, пошук і сортування

Клієнту рідко потрібні **всі** записи — дамо йому змогу звужувати вибірку через query-параметри
(`?...` у URL).

```bash
pip install django-filter
```
```python
# settings.py
REST_FRAMEWORK = {
    "DEFAULT_FILTER_BACKENDS": [
        "django_filters.rest_framework.DjangoFilterBackend",  # фільтр за полями
        "rest_framework.filters.SearchFilter",                # пошук
        "rest_framework.filters.OrderingFilter",              # сортування
    ],
}
```
```python
# articles/views.py
from django_filters.rest_framework import DjangoFilterBackend
from rest_framework import filters, viewsets

class ArticleViewSet(viewsets.ModelViewSet):
    queryset = Article.objects.all()
    serializer_class = ArticleSerializer

    filterset_fields = ["is_published"]        # ?is_published=true
    search_fields = ["title", "content"]       # ?search=python  (пошук по тексту)
    ordering_fields = ["created_at", "title"]  # ?ordering=-created_at
```

Приклади запитів:
```
GET /api/articles/?is_published=true          # лише опубліковані
GET /api/articles/?search=python              # де в title/content є "python"
GET /api/articles/?ordering=-created_at       # найновіші спершу ("-" = спадання)
GET /api/articles/?is_published=true&search=drf&ordering=title   # усе разом
```

> 🔎 **Фільтр vs пошук vs сортування:** *фільтр* — точний збіг за полем (`is_published=true`);
> *пошук* — часткове входження тексту в задані поля; *сортування* — порядок видачі. Їх комбінують.

## 4. Пагінація

Віддавати 10 000 записів одним запитом — погано (повільно, важко). **Пагінація** ріже видачу на
сторінки. Вмикається глобально в `settings.py`:

```python
REST_FRAMEWORK = {
    "DEFAULT_PAGINATION_CLASS": "rest_framework.pagination.PageNumberPagination",
    "PAGE_SIZE": 10,       # по 10 записів на сторінку
}
```

Тепер відповідь-список стає «обгорнутою»:
```json
{
  "count": 57,
  "next": "http://127.0.0.1:8000/api/articles/?page=2",
  "previous": null,
  "results": [ { "id": 1, "...": "..." } ]
}
```
Клієнт гортає сторінки через `?page=2`, `?page=3`, ...

| Клас пагінації | Параметри URL | Коли |
|----------------|---------------|------|
| `PageNumberPagination` | `?page=2` | прості «сторінки» (найчастіше) |
| `LimitOffsetPagination` | `?limit=10&offset=20` | коли треба довільний зсув |
| `CursorPagination` | `?cursor=...` | великі/стрічкові дані, стабільний порядок |

## 5. Автентифікація: хто ти?

**Автентифікація (authentication)** — встановити, **хто** робить запит. Не плутати з
**авторизацією / permissions** — «що тобі **можна**» (розділ 7).

DRF підтримує кілька схем:

| Схема | Як працює | Коли доречно |
|-------|-----------|--------------|
| **Session** | cookie сесії (як у звичайних сайтів Django) | коли фронтенд і бекенд на одному домені |
| **Basic** | логін:пароль у кожному запиті (base64) | лише для тестів/локально (небезпечно без HTTPS) |
| **Token** | клієнт шле сталий токен у заголовку | SPA, мобільні застосунки |
| **JWT** | самодостатній підписаний токен (бонус) | сучасний стандарт для stateless-API |

> 🔎 **Чому stateless-API люблять токени.** REST — **stateless**: сервер не тримає «сесію кроків».
> Тому клієнт додає підтвердження особи в **кожен** запит — зазвичай заголовок
> `Authorization: Token <ключ>`. Сесійні cookie теж працюють, але гірше пасують до мобільних/SPA.

## 6. Token Authentication на практиці

```python
# settings.py
INSTALLED_APPS = [
    # ...
    "rest_framework",
    "rest_framework.authtoken",     # ← додаємо
]

REST_FRAMEWORK = {
    "DEFAULT_AUTHENTICATION_CLASSES": [
        "rest_framework.authentication.TokenAuthentication",
    ],
    "DEFAULT_PERMISSION_CLASSES": [
        "rest_framework.permissions.IsAuthenticated",   # за замовчуванням — лише свої
    ],
}
```
```bash
python manage.py migrate          # створить таблицю токенів
```

Видати токен користувачу можна вбудованим ендпоінтом:
```python
# myproject/urls.py
from rest_framework.authtoken.views import obtain_auth_token
from django.urls import path

urlpatterns = [
    path("api/token/", obtain_auth_token),   # POST {username, password} -> {"token": "..."}
]
```

Використання:
```bash
# 1. Отримати токен (логін + пароль)
curl -X POST http://127.0.0.1:8000/api/token/ -d "username=ivan&password=secret"
# -> {"token": "9a8b7c6d..."}

# 2. Ходити з токеном у захищені ендпоінти
curl http://127.0.0.1:8000/api/articles/ -H "Authorization: Token 9a8b7c6d..."
```
Без валідного токена захищений ендпоінт поверне `401 Unauthorized`.

## 7. Permissions: що тобі можна?

**Permissions** визначають, чи має право саме **цей** користувач на **цю** дію. Вбудовані:

| Permission | Дозволяє |
|------------|----------|
| `AllowAny` | усім (навіть анонімам) |
| `IsAuthenticated` | лише автентифікованим |
| `IsAdminUser` | лише staff/адмінам |
| `IsAuthenticatedOrReadOnly` | читати всім, змінювати — лише автентифікованим |

Задають на рівні view/viewset:
```python
from rest_framework.permissions import IsAuthenticatedOrReadOnly

class ArticleViewSet(viewsets.ModelViewSet):
    queryset = Article.objects.all()
    serializer_class = ArticleSerializer
    permission_classes = [IsAuthenticatedOrReadOnly]   # GET — всім, зміни — своїм
```

### Кастомний permission
Напр. «редагувати статтю може лише її автор» (припустимо, у моделі є поле `author`):
```python
from rest_framework import permissions

class IsAuthorOrReadOnly(permissions.BasePermission):
    def has_object_permission(self, request, view, obj):
        if request.method in permissions.SAFE_METHODS:   # GET/HEAD/OPTIONS — читати всім
            return True
        return obj.author == request.user                # писати — лише автору
```

> 🎯 **Питання: різниця автентифікації й авторизації?** *Автентифікація* — **хто ти** (перевірка
> особи, напр. токен). *Авторизація / permissions* — **що тобі можна** (перевірка прав). Спершу
> йде автентифікація, потім — перевірка permission. Звідси й коди: `401` (не автентифікований)
> vs `403` (автентифікований, але без прав).

## 8. Обробка помилок і статус-коди

DRF сам повертає осмислені помилки, але важливо знати, **який код** коли доречний:

| Ситуація | Код | Приклад тіла |
|----------|-----|--------------|
| Успіх (список/об'єкт) | `200 OK` | дані |
| Створено | `201 Created` | створений об'єкт |
| Успіх без тіла (DELETE) | `204 No Content` | — |
| Невалідні дані | `400 Bad Request` | `{"title": ["...закороткий"]}` |
| Не автентифікований | `401 Unauthorized` | `{"detail": "Authentication credentials were not provided."}` |
| Немає прав | `403 Forbidden` | `{"detail": "You do not have permission..."}` |
| Немає об'єкта | `404 Not Found` | `{"detail": "Not found."}` |

Свою помилку кидають через виняток або `Response(..., status=...)`:
```python
from rest_framework.exceptions import ValidationError
from rest_framework.response import Response
from rest_framework import status

# варіант 1 — виняток (DRF сам зробить 400 з цим текстом):
raise ValidationError("Дата не може бути в минулому.")

# варіант 2 — явна відповідь:
return Response({"error": "Товар закінчився"}, status=status.HTTP_409_CONFLICT)
```

> 🔎 **Правило:** `4xx` — «винен клієнт» (не так надіслав, немає прав, немає ресурсу), `5xx` —
> «винен сервер» (баг, впала БД). Ніколи не повертай `200` з помилкою всередині — код статусу
> має чесно відображати результат.

## 9. 🎯 Що спитають на співбесіді (Урок 2)

| Питання | Коротка відповідь |
|---------|-------------------|
| ViewSet vs APIView? | ViewSet+Router = увесь CRUD і авто-маршрути; APIView = повний контроль |
| Що дає `ModelViewSet`? | Одразу 6 CRUD-дій (list/create/retrieve/update/partial_update/destroy) |
| Навіщо Router? | Автоматично генерує URL для ViewSet |
| Автентифікація vs авторизація? | Хто ти (401) vs що тобі можна (403) |
| Session vs Token auth? | Cookie-сесія (один домен) vs токен у заголовку (SPA/мобільні) |
| Навіщо пагінація? | Не віддавати тисячі записів одразу — швидкість і навантаження |
| `IsAuthenticatedOrReadOnly`? | Читання — всім, зміни — лише автентифікованим |
| Як зробити свій permission? | Клас від `BasePermission` + `has_permission`/`has_object_permission` |

# ✅ Підсумок уроку
- **ViewSet/ModelViewSet** — увесь CRUD в одному класі; дії `list/create/retrieve/update/partial_update/destroy` (+ власні через `@action`).
- **Router** (`DefaultRouter`/`SimpleRouter`) — автоматично генерує всі URL.
- **Фільтрація/пошук/сортування** — через `filterset_fields`, `search_fields`, `ordering_fields` (query-параметри `?...`).
- **Пагінація** — ріже видачу на сторінки (`PAGE_SIZE`); відповідь містить `count/next/previous/results`.
- **Автентифікація** (хто ти: Session/Basic/**Token**) ≠ **permissions** (що можна: `IsAuthenticated`, кастомні).
- **Status codes** мають чесно відображати результат: `2xx` успіх, `4xx` винен клієнт, `5xx` винен сервер.

### 📝 Домашнє завдання
[domashnie-zavdannia-lektsiya-2.ipynb](domashnie-zavdannia-lektsiya-2.ipynb)

### ▶️ Далі / бонус
За бажанням — **[бонус: JWT, Swagger, CORS](bonus-jwt-swagger-cors.ipynb)** (сучасна автентифікація
токенами, авто-документація API та доступ із фронтенду на іншому домені).

### 📚 Хочу знати більше
- ViewSets: <https://www.django-rest-framework.org/api-guide/viewsets/>
- Routers: <https://www.django-rest-framework.org/api-guide/routers/>
- Authentication: <https://www.django-rest-framework.org/api-guide/authentication/>
- Permissions: <https://www.django-rest-framework.org/api-guide/permissions/>